# Yearly AMR Pair Counts from PubMed Dates

This notebook addresses the main limitation of the interval-based AMR workflow: the current positive dataset is stored in 5-year bins, while the scientific-interest analysis would be cleaner on true publication years.

The workflow here is:

1. load the current `pmids_results_positive.parquet`
2. explode interval-level PMID lists into one row per `(pathogen, antibiotic, pmid)` association
3. query PubMed `esummary` to recover publication dates for each PMID
4. cache those PMID-level metadata locally so the process can be resumed
5. save both:
   - a pair-PMID association table with publication dates
   - a yearly AMR count table aggregated by `(pathogen, antibiotic, year)`


In [4]:
from pathlib import Path
import os

import pandas as pd

from amr.pubmed.yearly import (
    explode_positive_pair_pmids,
    load_pubmed_metadata_cache,
    refresh_pubmed_metadata_cache,
    attach_pubmed_metadata,
    aggregate_pair_counts_by_year,
    summarize_metadata_coverage,
)

DATA_DIR = Path(".")
POSITIVE_PATH = DATA_DIR / "pmids_results_positive.parquet"
CACHE_PATH = DATA_DIR / "pubmed_pmid_metadata.parquet"
PAIR_PMID_PATH = DATA_DIR / "pmids_results_positive_pair_pmid.parquet"
YEARLY_PATH = DATA_DIR / "pmids_results_positive_yearly.parquet"

PUBMED_EMAIL = os.environ.get("PUBMED_EMAIL", "xfcosta@gmail.com")
BATCH_SIZE = 200
CHECKPOINT_EVERY_BATCHES = 10

POSITIVE_PATH


PosixPath('pmids_results_positive.parquet')

## Load the current positive dataset

The existing positive parquet stores one row per `(pathogen, antibiotic, 5-year interval)` and keeps the matching PMIDs as a list. We first expand that into a true pair-PMID table.


In [5]:
positive_df = pd.read_parquet(POSITIVE_PATH)
pair_pmid_df = explode_positive_pair_pmids(positive_df)

print(f"interval rows: {len(positive_df):,}")
print(f"pair-PMID rows: {len(pair_pmid_df):,}")
print(f"unique PMIDs: {pair_pmid_df['pmid'].nunique():,}")

pair_pmid_df.head()


interval rows: 24,049
pair-PMID rows: 154,906
unique PMIDs: 63,316


,pathogen,antibiotic,start_date,end_date,pmid
0,Abiotrophia defectiva,Cefepime,2015/01/01,2019/12/31,27216060
1,Abiotrophia defectiva,Cefotaxime,2015/01/01,2019/12/31,27216060
2,Abiotrophia defectiva,Ceftriaxone,2010/01/01,2014/12/31,24034902
3,Abiotrophia defectiva,Ciprofloxacin,2010/01/01,2014/12/31,24034902
4,Abiotrophia defectiva,Clindamycin,2000/01/01,2004/12/31,10681362


## Fetch and cache PubMed dates

The helper below looks for an existing PMID metadata cache first. Only PMIDs that are still missing will be queried from PubMed.

Set a real email before running the fetch step, for example:

```bash
export PUBMED_EMAIL="your.name@example.com"
```

NCBI expects a real contact email for automated E-utilities requests.


In [6]:
cache_df = load_pubmed_metadata_cache(CACHE_PATH)
requested_pmids = pair_pmid_df["pmid"].astype(str).unique()
missing_pmids = sorted(set(requested_pmids) - set(cache_df["pmid"].astype(str)))

print(f"cached PMIDs: {cache_df['pmid'].nunique():,}")
print(f"missing PMIDs: {len(missing_pmids):,}")

if missing_pmids and PUBMED_EMAIL == "your_email@example.com":
    raise ValueError(
        "Set PUBMED_EMAIL to a real email address before fetching missing PubMed metadata."
    )

if missing_pmids:
    cache_df = refresh_pubmed_metadata_cache(
        requested_pmids,
        cache_path=CACHE_PATH,
        email=PUBMED_EMAIL,
        batch_size=BATCH_SIZE,
        checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
    )

cache_df.head()


cached PMIDs: 0
missing PMIDs: 63,316
Fetching PubMed metadata for 63316 missing PMIDs (0 already cached).
Fetched batch 1/317 (200 PMIDs).
Fetched batch 2/317 (200 PMIDs).
Fetched batch 3/317 (200 PMIDs).
Fetched batch 4/317 (200 PMIDs).
Fetched batch 5/317 (200 PMIDs).
Fetched batch 6/317 (200 PMIDs).
Fetched batch 7/317 (200 PMIDs).
Fetched batch 8/317 (200 PMIDs).
Fetched batch 9/317 (200 PMIDs).
Fetched batch 10/317 (200 PMIDs).
Checkpointed PubMed metadata cache to pubmed_pmid_metadata.parquet.


/mnt/data/sync/Projects/Pathogen Atlas/Antimicrobial Resistance/amr/pubmed/yearly.py:136: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  checkpoint_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)


Fetched batch 11/317 (200 PMIDs).
Fetched batch 12/317 (200 PMIDs).
Fetched batch 13/317 (200 PMIDs).
Fetched batch 14/317 (200 PMIDs).
Fetched batch 15/317 (200 PMIDs).
Fetched batch 16/317 (200 PMIDs).
Fetched batch 17/317 (200 PMIDs).
Fetched batch 18/317 (200 PMIDs).
Fetched batch 19/317 (200 PMIDs).
Fetched batch 20/317 (200 PMIDs).
Checkpointed PubMed metadata cache to pubmed_pmid_metadata.parquet.
Fetched batch 21/317 (200 PMIDs).
Fetched batch 22/317 (200 PMIDs).
Fetched batch 23/317 (200 PMIDs).
Fetched batch 24/317 (200 PMIDs).
Fetched batch 25/317 (200 PMIDs).
Fetched batch 26/317 (200 PMIDs).
Fetched batch 27/317 (200 PMIDs).
Fetched batch 28/317 (200 PMIDs).
Fetched batch 29/317 (200 PMIDs).
Fetched batch 30/317 (200 PMIDs).
Checkpointed PubMed metadata cache to pubmed_pmid_metadata.parquet.
Fetched batch 31/317 (200 PMIDs).
Fetched batch 32/317 (200 PMIDs).
Fetched batch 33/317 (200 PMIDs).
Fetched batch 34/317 (200 PMIDs).
Fetched batch 35/317 (200 PMIDs).
Fetched batch 

,pmid,pub_date,pub_year,venue
0,1000196,1976-11-01,1976,British medical journal
1,100029,1978-08-01,1978,The American review of respiratory disease
2,100050,1978-08-01,1978,Antimicrobial agents and chemotherapy
3,100051,1978-08-01,1978,Antimicrobial agents and chemotherapy
4,100194,1978-07-01,1978,Canadian journal of ophthalmology. Journal can...


## Save the pair-PMID association table

This table is the key bridge from interval-level search results to yearly AMR counts. Each row represents one positive PMID associated with one `(pathogen, antibiotic)` pair, now annotated with PubMed date metadata.


In [7]:
pair_pmid_enriched_df = attach_pubmed_metadata(pair_pmid_df, cache_df)
pair_pmid_enriched_df.to_parquet(PAIR_PMID_PATH, index=False)

print(f"saved pair-PMID associations to {PAIR_PMID_PATH}")
summarize_metadata_coverage(pair_pmid_enriched_df)


saved pair-PMID associations to pmids_results_positive_pair_pmid.parquet


,metric,value
0,pair_pmid_rows,154906
1,unique_pmids,63316
2,rows_with_pub_year,154906
3,rows_missing_pub_year,0
4,rows_pub_year_matches_interval,154897
5,rows_pub_year_outside_interval,9


## Aggregate to yearly AMR counts

The yearly table below groups the enriched pair-PMID rows by publication year, giving a true annual count rather than a 5-year interval count.


In [8]:
yearly_df = aggregate_pair_counts_by_year(pair_pmid_enriched_df)
yearly_df.to_parquet(YEARLY_PATH, index=False)

print(f"saved yearly AMR counts to {YEARLY_PATH}")
print(f"yearly rows: {len(yearly_df):,}")

yearly_df.head()


saved yearly AMR counts to pmids_results_positive_yearly.parquet
yearly rows: 45,244


,pathogen,antibiotic,year,count,pmids
0,Abiotrophia defectiva,Cefepime,2016,1,[27216060]
1,Abiotrophia defectiva,Cefotaxime,2016,1,[27216060]
2,Abiotrophia defectiva,Ceftriaxone,2013,1,[24034902]
3,Abiotrophia defectiva,Ciprofloxacin,2013,1,[24034902]
4,Abiotrophia defectiva,Clindamycin,2000,1,[10681362]


## Quick diagnostics

This is a useful sanity check after the PubMed metadata pass. In most cases, a PMID should land inside the 5-year interval where it was originally found. Any rows outside that interval are worth reviewing.


In [9]:
diagnostics = summarize_metadata_coverage(pair_pmid_enriched_df)
diagnostics

pair_pmid_enriched_df.loc[
    pair_pmid_enriched_df["pub_year_matches_interval"] == False,
    ["pathogen", "antibiotic", "pmid", "pub_year", "start_date", "end_date", "venue"],
].head(20)


,pathogen,antibiotic,pmid,pub_year,start_date,end_date,venue
9531,Bordetella pertussis,Erythromycin,39313898,2025,2020/01/01,2024/12/31,Current pharmaceutical biotechnology
29470,Enterococcus gallinarum,Erythromycin,39046670,2025,2020/01/01,2024/12/31,Probiotics and antimicrobial proteins
37176,Escherichia coli,Cefotaxime,39461740,2026,2020/01/01,2024/12/31,BMJ military health
86027,Mycobacterium tuberculosis,Rifampicin,39058037,2025,2020/01/01,2024/12/31,Monaldi archives for chest disease = Archivio ...
135351,Staphylococcus aureus,Vancomycin,37886812,2026,2020/01/01,2024/12/31,The international journal of lower extremity w...
138961,Staphylococcus pseudintermedius,Amoxicillin,37855899,2026,2020/01/01,2024/12/31,"Microbial drug resistance (Larchmont, N.Y.)"
138969,Staphylococcus pseudintermedius,Amoxicillin/clavulanic-acid,37855899,2026,2020/01/01,2024/12/31,"Microbial drug resistance (Larchmont, N.Y.)"
138994,Staphylococcus pseudintermedius,Cefazolin,37855899,2026,2020/01/01,2024/12/31,"Microbial drug resistance (Larchmont, N.Y.)"
139372,Staphylococcus pseudintermedius,Tetracycline,37855899,2026,2020/01/01,2024/12/31,"Microbial drug resistance (Larchmont, N.Y.)"
